In [1]:

!pip install fugashi[unidic-lite] -q

import os, glob, re
from collections import Counter
import numpy as np
import pandas as pd
from fugashi import Tagger


!wget -q https://www.rondhuit.com/download/ldcc-20140209.tar.gz
!tar -xvzf ldcc-20140209.tar.gz > /dev/null

os.makedirs("/content/japanese_corpus", exist_ok=True)
raw_file = "/content/japanese_corpus/livedoor_news.txt"

with open(raw_file, "w", encoding="utf-8") as out:
    for file in glob.glob("text/**/*.txt", recursive=True):
        with open(file, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()[2:]  # bỏ header 2 dòng đầu
            content = " ".join([line.strip() for line in lines if line.strip()])
            out.write(content + "\n")

print("✅ Đã gom dữ liệu vào:", raw_file)


tagger = Tagger()
clean_file = "/content/japanese_corpus/livedoor_news_tokenized.txt"

with open(raw_file, "r", encoding="utf-8") as inp, \
     open(clean_file, "w", encoding="utf-8") as out:
    for line in inp:
        tokens = [word.surface for word in tagger(line)]
        out.write(" ".join(tokens) + "\n")

print("✅ Đã tokenize xong")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 15.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.9/697.9 kB 6.4 MB/s eta 0:00:00
✅ Đã gom dữ liệu vào: /content/japanese_corpus/livedoor_news.txt
✅ Đã tokenize xong


In [18]:

!pip install fugashi[unidic-lite] -q

import os, json
from collections import Counter
from fugashi import Tagger


def is_kanji(word):
    """Kiểm tra từ toàn bộ là Kanji"""
    return all('\u4e00' <= c <= '\u9fff' for c in word)


clean_file = "/content/japanese_corpus/livedoor_news_tokenized.txt"

with open(clean_file, "r", encoding="utf-8") as f:
    lines = f.readlines()


tagger = Tagger()

# Tokenize và tách từng chữ Kanji riêng
kanji_tokens_list = []
for line in lines:
    tokens = []
    for word in tagger(line):
        for c in word.surface:
            if is_kanji(c):
                tokens.append(c)
    if tokens:
        kanji_tokens_list.append(tokens)

print("✅ Token Kanji (từng chữ) từ corpus:", len(kanji_tokens_list))


def generate_ngrams(tokens, n=2):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

min_n, max_n = 2, 4
ngram_counter = Counter()

for tokens in kanji_tokens_list:
    for n in range(min_n, max_n+1):
        ngrams = generate_ngrams(tokens, n)
        ngram_counter.update(ngrams)

print("✅ Đã tạo n-gram (2-4 chữ) từ token Kanji")


jmdict_dir = "/content/drive/MyDrive/Dataset_Kanji_2/Dataset_Leaf/JMdict_english"

term_entries = []
for fname in os.listdir(jmdict_dir):
    if fname.startswith("term_bank") and fname.endswith(".json"):
        with open(os.path.join(jmdict_dir, fname), "r", encoding="utf-8") as f:
            term_entries.extend(json.load(f))

# Tạo set các từ để tra cứu nhanh
all_terms = set(entry[0] for entry in term_entries)  # cột 0 chứa từ Kanji/Kana


def filter_valid_words(words):
    """Lọc ra những từ ghép có nghĩa dựa trên term_bank JSON"""
    return [w for w in words if w in all_terms]


def suggest_compounds(kanji, top_n=20, min_freq=2):
    filtered_ngrams = {}
    for ngram, count in ngram_counter.items():
        if kanji in ngram and count >= min_freq:
            filtered_ngrams[ngram] = count

    sorted_ngrams = sorted(filtered_ngrams.items(), key=lambda x: x[1], reverse=True)

    top_ngrams = sorted_ngrams[:top_n]

    top_words = ["".join(ngram) for ngram, _ in top_ngrams]

    return top_words


✅ Token Kanji (từng chữ) từ corpus: 7378
✅ Đã tạo n-gram (2-4 chữ) từ token Kanji


In [20]:

kanji = "年"

top_compounds = suggest_compounds(kanji, top_n=10, min_freq=2)

valid_compounds = filter_valid_words(top_compounds)

print("➡ Top từ ghép Kanji có nghĩa:")
print(valid_compounds)

➡ Top từ ghép Kanji có nghĩa:
['年月', '年月日', '今年', '昨年', '年収', '年間', '年齢']


In [21]:
import pickle

with open("/content/japanese_corpus/kanji_tokens_list.pkl", "wb") as f:
    pickle.dump(kanji_tokens_list, f)


In [22]:
with open("/content/japanese_corpus/ngram_counter.pkl", "wb") as f:
    pickle.dump(ngram_counter, f)


In [23]:
with open("/content/japanese_corpus/all_terms.pkl", "wb") as f:
    pickle.dump(all_terms, f)
